In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path("..").resolve()))

In [4]:
from src.reasoning_compression.features import (
    parse_response,
    count_tokens,
    count_tokens_batch,
    extract_problem_features,
    build_feature_tables,
    extract_spacy_problem_features,
)

N_ROWS = 500

# Responses feature extraction

This section parses the model response into its main components: the hidden reasoning phase, the final answer, and the aligned sequence of reasoning blocks and summaries.

The goal is to convert each raw response string into structured trace-level and block-level features. At the trace level, we measure the overall length of the reasoning process, answer, and full response. At the block level, each `(block, summary)` pair is treated as one observed compression event, allowing us to measure token-based compression ratios and define the initial classification target.

Token counts are computed using `tiktoken` with the `cl100k_base` BPE encoding. This provides a consistent BPE-based approximation of LLM tokenization, but may not exactly match the tokenizer of every model used to generate OpenMementos.

For exact tokenization for the generating model we should use Hugging Face transformers:

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("MODEL_NAME")

def count_tokens(text: str) -> int:
    if text is None:
        return 0
    return len(tokenizer.encode(text, add_special_tokens=False))

    

## Engineered Output Tables

The `build_feature_tables` function returns two dataframes:

- `df_traces`: one row per original reasoning trace.
- `df_blocks`: one row per aligned `(reasoning block, summary)` pair.

`df_traces` is useful for trace-level analysis, such as comparing total reasoning length, number of blocks, or answer length across domains and sources.

`df_blocks` is the main table for compression modeling. Each row represents one compression event: a full reasoning block and the shorter summary generated for that block.

### Problem-derived variables

These variables are computed from the original `problem` text and are valid predictors because they are available before the model produces its reasoning:

- `problem_tokens`
- `problem_chars`
- `problem_math_symbol_share`
- `problem_has_multiple_choice`
- `problem_has_code_fence`
- `problem_question_mark_count`

### Response-derived variables

These variables are computed from the model `response`, including the parsed reasoning trace, block summaries, and final answer:

- `response_tokens`
- `think_tokens`
- `answer_tokens`
- `n_blocks`
- `n_summaries`
- `block_summary_delta`
- `block_index`
- `relative_block_position`
- `block_tokens`
- `summary_tokens`
- `summary_to_block_token_ratio`
- `token_compression_savings`

The compression ratio and compression savings variables are outcomes, not predictors. They are used to define and evaluate compression behavior, but they should not be included as input features when training models to predict compressibility.

Character-based features are retained in the engineered tables as diagnostic variables, token-based features are the core measurement layer for compression and modeling. 


In [5]:
# check dataframe sizes

df_traces, df_blocks = build_feature_tables(N_ROWS)

df_traces.shape, df_blocks.shape

((500, 19), (4735, 21))

In [6]:
random_trace_id = int(df_traces["trace_id"].sample(1, random_state=42).iloc[0])
df_traces.query("trace_id == @random_trace_id")

,trace_id,domain,source,difficulty,problem_chars,problem_tokens,problem_math_symbol_share,problem_has_multiple_choice,problem_has_code_fence,problem_question_mark_count,response_chars,response_tokens,think_chars,think_tokens,answer_chars,answer_tokens,n_blocks,n_summaries,block_summary_delta
361,361,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,0,44760,14983,43399,14581,1346,398,11,11,0


In [7]:
df_blocks.query("trace_id == @random_trace_id").sort_values("block_index")

,trace_id,block_index,domain,source,difficulty,problem_chars,problem_tokens,problem_math_symbol_share,problem_has_multiple_choice,problem_has_code_fence,...,n_blocks_in_trace,relative_block_position,block_chars,summary_chars,block_tokens,summary_tokens,summary_to_block_char_ratio,summary_to_block_token_ratio,char_compression_savings,token_compression_savings
3397,361,0,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.0,2418,317,633,90,0.131100,0.142180,0.868900,0.857820
3398,361,1,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.1,1907,608,639,238,0.318825,0.372457,0.681175,0.627543
3399,361,2,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.2,3445,443,1149,187,0.128592,0.162750,0.871408,0.837250
3400,361,3,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.3,4267,1337,1304,444,0.313335,0.340491,0.686665,0.659509
3401,361,4,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.4,4012,940,1248,333,0.234297,0.266827,0.765703,0.733173
3402,361,5,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.5,3996,444,1264,130,0.111111,0.102848,0.888889,0.897152
3403,361,6,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.6,3668,444,1171,149,0.121047,0.127242,0.878953,0.872758
3404,361,7,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.7,2411,504,795,169,0.209042,0.212579,0.790958,0.787421
3405,361,8,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.8,7585,628,2790,215,0.082795,0.077061,0.917205,0.922939
3406,361,9,code,nvidia/OpenCodeReasoning,8,1833,547,0.030551,0,0,...,11,0.9,1255,696,548,334,0.554582,0.609489,0.445418,0.390511


`df_traces`: one row per original reasoning trace.

`df_traces` is useful for trace-level analysis, such as comparing total reasoning length, number of blocks, or answer length across domains and sources.

In [6]:
df_traces.head()

,trace_id,domain,source,difficulty,problem_chars,problem_tokens,problem_math_symbol_share,problem_has_multiple_choice,problem_has_code_fence,problem_question_mark_count,response_chars,response_tokens,think_chars,think_tokens,answer_chars,answer_tokens,n_blocks,n_summaries,block_summary_delta
0,0,code,stackexchange_codegolf,7,931,227,0.018260,0,0,0,17027,4777,14971,4309,2041,465,7,7,0
1,1,code,stackexchange_codegolf,7,780,148,0.000000,0,0,0,15774,3482,13694,2983,2065,496,8,8,0
2,2,code,stackexchange_codegolf,8,2919,955,0.055841,0,0,2,48716,15695,46465,15166,2236,526,4,4,0
3,3,code,nvidia/OpenCodeReasoning,8,2620,546,0.007634,0,0,0,24861,6742,21086,5856,3760,883,5,5,0
4,4,code,stackexchange_codegolf,7,604,145,0.033113,0,0,2,24557,5488,20994,4584,3548,901,10,10,0


`df_blocks`: one row per aligned `(reasoning block, summary)` pair.

`df_blocks` is the main table for compression modeling. Each row represents one compression event: a full reasoning block and the shorter summary generated for that block.


In [7]:
df_blocks.head()

,trace_id,block_index,domain,source,difficulty,problem_chars,problem_tokens,problem_math_symbol_share,problem_has_multiple_choice,problem_has_code_fence,...,n_blocks_in_trace,relative_block_position,block_chars,summary_chars,block_tokens,summary_tokens,summary_to_block_char_ratio,summary_to_block_token_ratio,char_compression_savings,token_compression_savings
0,0,0,code,stackexchange_codegolf,7,931,227,0.01826,0,0,...,7,0.000000,809,319,200,77,0.394314,0.385000,0.605686,0.615000
1,0,1,code,stackexchange_codegolf,7,931,227,0.01826,0,0,...,7,0.166667,2608,445,737,111,0.170629,0.150611,0.829371,0.849389
2,0,2,code,stackexchange_codegolf,7,931,227,0.01826,0,0,...,7,0.333333,2213,465,654,133,0.210122,0.203364,0.789878,0.796636
3,0,3,code,stackexchange_codegolf,7,931,227,0.01826,0,0,...,7,0.500000,1067,328,342,92,0.307404,0.269006,0.692596,0.730994
4,0,4,code,stackexchange_codegolf,7,931,227,0.01826,0,0,...,7,0.666667,1340,354,385,117,0.264179,0.303896,0.735821,0.696104


In [8]:
df_traces.isna().sum()

trace_id                       0
domain                         0
source                         0
difficulty                     0
problem_chars                  0
problem_tokens                 0
problem_math_symbol_share      0
problem_has_multiple_choice    0
problem_has_code_fence         0
problem_question_mark_count    0
response_chars                 0
response_tokens                0
think_chars                    0
think_tokens                   0
answer_chars                   0
answer_tokens                  0
n_blocks                       0
n_summaries                    0
block_summary_delta            0
dtype: int64

In [9]:
df_blocks.isna().sum()

trace_id                        0
block_index                     0
domain                          0
source                          0
difficulty                      0
problem_chars                   0
problem_tokens                  0
problem_math_symbol_share       0
problem_has_multiple_choice     0
problem_has_code_fence          0
problem_question_mark_count     0
n_blocks_in_trace               0
relative_block_position         0
block_chars                     0
summary_chars                   0
block_tokens                    0
summary_tokens                  0
summary_to_block_char_ratio     0
summary_to_block_token_ratio    0
char_compression_savings        0
token_compression_savings       0
dtype: int64

In [10]:
summary_df_blocks = (
    df_blocks[[
        "block_tokens",
        "summary_tokens",
        "token_compression_savings",
    ]]
    .rename(columns={"token_compression_savings": "token_compression"})
    .describe()
)

summary_df_blocks_display = summary_df_blocks.map(
    lambda x: f"{int(x)}" if float(x).is_integer() else f"{x:.4f}"
)

summary_df_blocks_display.loc[
    ["count", "mean", "std"],
    "token_compression",
] = "-"

summary_df_blocks_display

,block_tokens,summary_tokens,token_compression
count,9449,9449,-
mean,1155.1374,194.6209,-
std,665.0086,110.5515,-
min,200,15,-0.2899
25%,664,116,0.7232
50%,1038,174,0.8236
75%,1500,246,0.8921
max,9906,1059,0.9869


Across 9,449 blocks, the summaries substantially reduce token usage. The original blocks have a median length of 1,038 tokens, while the summaries have a median length of only 174 tokens. In other words, a typical summary is much shorter than the source block.
The token_compression values suggest strong compression overall. The median compression saving is 0.8236, meaning the typical summary saves about 82% of the original token count. The middle half of cases save between roughly 72% and 89%, so compression is consistently high for most blocks.
There is one important caveat: the minimum compression value is -0.2899, meaning at least one summary is longer than its original block by about 29%. That case is worth inspecting, as it may reflect a short source block, an overly verbose summary, or a malformed summarisation output. Overall, though, the summaries appear to be highly effective at reducing token volume.

## Initial Compression Classification Target

At this stage, we define a simple binary classification target at the block level.

Each row in `df_blocks` represents one reasoning block and its corresponding summary. The main compression quantity is:

`summary_to_block_token_ratio = summary_tokens / block_tokens`

Lower values mean that the summary is much shorter than the original reasoning block, so the block was compressed more aggressively.

To create an initial classification target, we label blocks in the lowest quartile of `summary_to_block_token_ratio` as high-compression examples:

`high_token_compression = 1` if the block is in the most-compressed 25% of blocks.

This target is relative to the current sample, not an absolute external threshold. It is useful for baseline modeling because it creates a balanced enough binary task while preserving the interpretation that label `1` means unusually strong compression.

In [11]:
compression_threshold = df_blocks["summary_to_block_token_ratio"].quantile(0.25)

df_blocks["high_token_compression"] = (
    df_blocks["summary_to_block_token_ratio"] <= compression_threshold
).astype(int)

compression_threshold, df_blocks["high_token_compression"].value_counts(normalize=True)

(np.float64(0.1079429735234216),
 high_token_compression
 0    0.749921
 1    0.250079
 Name: proportion, dtype: float64)

In [12]:
feature_columns = [
    "block_index",
    "relative_block_position",
    "n_blocks_in_trace",
    "problem_tokens",
    "block_tokens",
    "domain",
    "source",
    "difficulty",
]

target_column = "high_token_compression"

df_model = df_blocks[feature_columns + [target_column]].copy()

df_model.head()

,block_index,relative_block_position,n_blocks_in_trace,problem_tokens,block_tokens,domain,source,difficulty,high_token_compression
0,0,0.000000,7,227,200,code,stackexchange_codegolf,7,0
1,1,0.166667,7,227,737,code,stackexchange_codegolf,7,0
2,2,0.333333,7,227,654,code,stackexchange_codegolf,7,0
3,3,0.500000,7,227,342,code,stackexchange_codegolf,7,0
4,4,0.666667,7,227,385,code,stackexchange_codegolf,7,0


## Target Distribution

The table below shows how many block-level observations fall into each target class.

Because the target is defined using the 25th percentile of `summary_to_block_token_ratio`, we expect approximately one quarter of blocks to have `high_token_compression = 1`. Small deviations can occur when multiple rows have exactly the same ratio at the threshold.

In [13]:
high_compression_summary = (
    df_blocks["high_token_compression"]
    .value_counts()
    .rename_axis("high_token_compression")
    .reset_index(name="n_blocks")
)

high_compression_summary["share"] = (
    high_compression_summary["n_blocks"] / len(df_blocks)
).round(4)

high_compression_summary

,high_token_compression,n_blocks,share
0,0,7086,0.7499
1,1,2363,0.2501


The resulting class balance is suitable for an initial binary classification model. The majority class contains normally or weakly compressed blocks, while the minority class contains the most compressed blocks in the sampled data.

The target should not be interpreted as a universal definition of compression quality. It is an empirical label constructed from the current dataset sample and may be revised later using absolute token-ratio thresholds, task-specific thresholds, or outcome-based quality measures.

In [14]:
df_blocks["summary_to_block_token_ratio"].describe().round(4)

count    9449.0000
mean        0.2130
std         0.1462
min         0.0131
25%         0.1079
50%         0.1764
75%         0.2768
max         1.2899
Name: summary_to_block_token_ratio, dtype: float64

In [15]:
(df_blocks["summary_to_block_token_ratio"] > 1).sum()

np.int64(9)

### Compression Ratio Edge Cases

9 block-summary pairs have `summary_to_block_token_ratio > 1`, meaning the summary contains more tokens than the original block.
These cases are retained for now because they may reflect valid behavior, such as very short reasoning blocks, verbose summaries, or occasional formatting/parsing edge cases.
Before final modeling or reporting, check these rows to decide whether they are valid observations or should be filtered as anomalies.

## spaCy feature extensions

This section adds lightweight linguistic features from the original problem text using spaCy.

These features are optional and will be evaluated later to see whether they improve compressibility prediction. They are computed at the trace level because each problem appears once per reasoning trace, then merged onto the block-level modeling table through `trace_id`.

The features are valid predictors because they are derived only from the original problem text, not from the generated summary or compression outcome.

`spacy_sentence_count`: number of sentences in the problem.

`spacy_avg_sentence_tokens`: average number of tokens per sentence.

`spacy_stopword_share`: share of tokens that are common function words like “the”, “is”, “of”.

`spacy_punctuation_share`: share of tokens that are punctuation.

`spacy_numeric_token_share`: share of tokens that look numeric.

`spacy_oov_share`: share of tokens outside spaCy’s vocabulary.

`spacy_noun_share`: share of tokens tagged as nouns or proper nouns.

`spacy_verb_share`: share of tokens tagged as verbs or auxiliaries.

`spacy_entity_count`: number of named entities detected in the problem.


In [23]:
import spacy

In [24]:
nlp = spacy.load("en_core_web_sm")

In [25]:
def extract_spacy_problem_features(doc) -> dict:
    #ignore whitespaces
    tokens = [tok for tok in doc if not tok.is_space]

    # prevent zero division errors
    if not tokens:
        return {
            "spacy_sentence_count": 0,
            "spacy_avg_sentence_tokens": 0,
            "spacy_stopword_share": 0,
            "spacy_punctuation_share": 0,
            "spacy_numeric_token_share": 0,
            "spacy_oov_share": 0,
            "spacy_noun_share": 0,
            "spacy_verb_share": 0,
            "spacy_entity_count": 0,
        }

    sentence_lengths = [
        len([tok for tok in sent if not tok.is_space])
        for sent in doc.sents
    ]

    return {
        "spacy_sentence_count": len(sentence_lengths),
        "spacy_avg_sentence_tokens": (
            sum(sentence_lengths) / len(sentence_lengths)
            if sentence_lengths else 0
        ),

        # normalize counts by prompt length
        "spacy_stopword_share": sum(tok.is_stop for tok in tokens) / len(tokens),
        "spacy_punctuation_share": sum(tok.is_punct for tok in tokens) / len(tokens),
        "spacy_numeric_token_share": sum(tok.like_num for tok in tokens) / len(tokens),
        "spacy_oov_share": sum(tok.is_oov for tok in tokens) / len(tokens),
        # entity/object-heavy or action/relation-heavy prompts
        "spacy_noun_share": sum(tok.pos_ in {"NOUN", "PROPN"} for tok in tokens) / len(tokens),
        "spacy_verb_share": sum(tok.pos_ in {"VERB", "AUX"} for tok in tokens) / len(tokens),
        # entity counts
        "spacy_entity_count": len(doc.ents),
    }

In [26]:
# work on one problem string per trace 
# compute spaCy features once per trace rather than once per block

problem_text_by_trace = (
    df_traces[["trace_id"]]
    .copy()
)

# re-stream only the same dataset portion to recover the original problem text.
ds_stream = load_dataset(
    DATASET_ID,
    split=SPLIT,
    streaming=True,
)

problem_text_by_trace["problem"] = [
    row.get("problem") or ""
    for row in islice(ds_stream, len(df_traces))
]

In [27]:
spacy_rows = []

for trace_id, doc in zip(
    problem_text_by_trace["trace_id"],
    nlp.pipe(problem_text_by_trace["problem"], batch_size=64)
):
    spacy_rows.append({
        "trace_id": trace_id,
        **extract_spacy_problem_features(doc),
    })

df_spacy_problem_features = pd.DataFrame(spacy_rows)

df_spacy_problem_features.head()

,trace_id,spacy_sentence_count,spacy_avg_sentence_tokens,spacy_stopword_share,spacy_punctuation_share,spacy_numeric_token_share,spacy_oov_share,spacy_noun_share,spacy_verb_share,spacy_entity_count
0,0,10,20.400000,0.348039,0.132353,0.044118,1.0,0.254902,0.147059,15
1,1,7,20.571429,0.458333,0.104167,0.027778,1.0,0.250000,0.250000,7
2,2,23,26.695652,0.283388,0.219870,0.100977,1.0,0.247557,0.104235,74
3,3,23,21.608696,0.380282,0.082495,0.070423,1.0,0.291751,0.154930,21
4,4,8,16.625000,0.345865,0.180451,0.015038,1.0,0.255639,0.165414,6


In [29]:
summary = df_spacy_problem_features.describe()

summary.map(
    lambda x: f"{int(x)}" if float(x).is_integer() else f"{x:.4f}"
)

,trace_id,spacy_sentence_count,spacy_avg_sentence_tokens,spacy_stopword_share,spacy_punctuation_share,spacy_numeric_token_share,spacy_oov_share,spacy_noun_share,spacy_verb_share,spacy_entity_count
count,1000,1000,1000,1000,1000,1000,1000,1000,1000,1000
mean,499.5000,20.0520,25.6367,0.3457,0.1664,0.0953,1,0.2569,0.1255,35.3230
std,288.8194,17.7014,12.2564,0.0885,0.0842,0.0776,0,0.0584,0.0364,32.0386
min,0,1,5.3864,0.0118,0,0,1,0.0767,0.0024,0
25%,249.7500,11,19.9409,0.2899,0.1168,0.0422,1,0.2306,0.1028,17
50%,499.5000,17,22.7143,0.3533,0.1455,0.0910,1,0.2549,0.1240,29
75%,749.2500,25,27.8484,0.4094,0.1882,0.1223,1,0.2829,0.1478,43
max,999,249,187,0.5597,0.8433,0.7136,1,0.8570,0.2500,466


In [ ]:
# merge spaCy features onto the trace-level table


df_traces = df_traces.merge(
    df_spacy_problem_features,
    on="trace_id",
    how="left",
)

df_blocks = df_blocks.merge(
    df_spacy_problem_features,
    on="trace_id",
    how="left",
)

df_traces.shape, df_blocks.shape

# Engineered data storing 

Notebook 02:
raw streamed dataset -> feature engineering -> local ignored parquet

Notebook 03:
local ignored parquet -> modeling pipeline

In [31]:
from pathlib import Path

In [32]:
DATA_DIR = Path("../data")
DATA_DIR.mkdir(exist_ok=True)

df_traces.to_parquet(DATA_DIR / "traces_features_sample.parquet", index=False)
df_blocks.to_parquet(DATA_DIR / "blocks_features_sample.parquet", index=False)

In [33]:
list(DATA_DIR.glob("*.parquet"))

[PosixPath('../data/blocks_features_sample.parquet'),
 PosixPath('../data/traces_features_sample.parquet')]

In [34]:
df_blocks.shape

(9449, 22)